# OIM3600 — Data Structures, Design, and AI
## Class Notebook 1: Choose the Shape, Then Specify It


Today connects two ideas that belong together:

1. **The representation you choose changes the computation.**
2. **Any important decision you do not specify is a decision you hand to AI.**

The goal is not to describe Python line by line in English. The goal is to make the design decisions that matter before implementation begins.

We will work through:

- parallel lists and why they are fragile
- lists of dictionaries as records
- grids as lists of lists
- aliasing in nested mutable structures
- flat ↔ grid index arithmetic
- choosing representations deliberately
- incomplete vs. complete AI specifications
- distinguishing design decisions from implementation details

## Part 1 — Representation Drives Computation
**Target time: 10 minutes**

A **representation** is the shape we choose for information inside a program.

The same real-world information can often be represented in several ways. The important question is not:

> "Which structure is best?"

It is:

> **"Which structure makes the questions I need to answer easier and safer?"**

A representation is therefore a **design decision**, not a formatting decision.

In [ ]:
# PREDICT BEFORE RUNNING
#
# These two structures contain related information.
# Which one makes it easier to accidentally break the relationship?

names  = ["Ana", "Ben", "Chen"]
sales  = [1200, 900, 1500]

customers = [
    {"name": "Ana",  "sales": 1200},
    {"name": "Ben",  "sales": 900},
    {"name": "Chen", "sales": 1500},
]

print(names)
print(sales)
print(customers)

## Part 2 — The Parallel-Lists Trap
**Target time: 15 minutes**

With **parallel lists**, position carries meaning:

- `names[i]`
- `sales[i]`

must continue to describe the same person.

That creates an invariant:

> **All parallel lists must stay the same length and corresponding positions must continue to refer to the same record.**

The representation works, but it is easy to damage.

In [ ]:
# EXERCISE 1 — FIND THE BUG
#
# Predict what happens to the relationship between names and sales.

names = ["Ana", "Ben", "Chen"]
sales = [1200, 900, 1500]

names.sort()

print(names)
print(sales)

# Answer in comments:
# 1. Did either list become invalid by itself?
# 2. What relationship was broken?
# 3. Why is this a representation problem rather than a sorting problem?

## Part 3 — Records: A List of Dictionaries
**Target time: 25 minutes**

A common repair is to keep the fields describing one thing together.

A dictionary can represent one record:

```python
{"name": "Ana", "region": "East", "sales": 1200}
```

A list of dictionaries can then represent many records.

This is similar to a table:

- one dictionary → one row
- dictionary keys → field names / columns
- list → the collection of rows

In [ ]:
customers = [
    {"name": "Ana",  "region": "East",  "sales": 1200},
    {"name": "Ben",  "region": "West",  "sales": 900},
    {"name": "Chen", "region": "East",  "sales": 1500},
]

# Predict before running:
#
# 1. What is customers[1]?
# 2. What is customers[1]["name"]?
# 3. What does the loop variable customer refer to on each iteration?

print(customers[1])
print(customers[1]["name"])

for customer in customers:
    print(customer["name"], customer["sales"])

In [ ]:
# EXERCISE 2 — FILTER RECORDS
#
# Print the name and sales for customers in the East region.
#
# Then answer:
# Why does keeping each record together make this operation easier
# to reason about than parallel lists?

customers = [
    {"name": "Ana",  "region": "East",  "sales": 1200},
    {"name": "Ben",  "region": "West",  "sales": 900},
    {"name": "Chen", "region": "East",  "sales": 1500},
    {"name": "Dana", "region": "South", "sales": 1100},
]

# your code here

### Sorting records with `key=`

Python does not know how to order dictionaries directly.

Instead, a **key function** tells `sort()` which value from each record should determine the order.

In [ ]:
customers = [
    {"name": "Ana",  "region": "East", "sales": 1200},
    {"name": "Ben",  "region": "West", "sales": 900},
    {"name": "Chen", "region": "East", "sales": 1500},
]

def get_sales(customer):
    return customer["sales"]

# PREDICT BEFORE RUNNING:
# Which customer will appear first?

customers.sort(key=get_sales, reverse=True)

# customers.sort(key=(lambda customer: customer["sales"]), reverse=True)

for customer in customers:
    print(customer)

In [ ]:
# EXERCISE 3 — CHOOSE THE KEY
#
# Sort the records alphabetically by region.
# Within this exercise, do not change the records themselves.

products = [
    {"sku": "A14", "region": "West",  "price": 19.95},
    {"sku": "B07", "region": "East",  "price": 24.50},
    {"sku": "C22", "region": "South", "price": 11.25},
    {"sku": "D03", "region": "East",  "price": 32.00},
]

# your code here


# sort on two fields

## Part 4 — Grids: Lists of Lists
**Target time: 25 minutes**

Not all information is naturally a set of named records.

Some information is naturally arranged by **row and column**:

- seating charts
- game boards
- weekly-by-hour measurements
- matrices

A list of lists is a natural representation for this kind of data.

In [ ]:
seats = [
    ["open", "sold", "open", "open"],
    ["sold", "sold", "open", "open"],
    ["open", "open", "open", "sold"],
]

# Predict:
# What does seats[1] return?
# What does seats[1][2] return?

print(seats[1])
print(seats[1][2])

In [ ]:
# EXERCISE 4 — WALK THE GRID
#
# Print every location as:
# row 0 col 0 open
# row 0 col 1 sold
# ...

seats = [
    ["open", "sold", "open", "open"],
    ["sold", "sold", "open", "open"],
    ["open", "open", "open", "sold"],
]

# your code here


### The list-repetition aliasing trap

Nested mutable structures introduce another invariant:

> **Rows that are supposed to change independently must be different list objects.**

This is wrong:

```python
grid = [[0] * 4] * 3
```

The outer repetition repeats references to the **same row object**.

In [ ]:
# PREDICT BEFORE RUNNING

bad_grid = [[0] * 4] * 3
bad_grid[0][1] = 9

print(bad_grid)

# Now compare object identities.
print(id(bad_grid[0]))
print(id(bad_grid[1]))
print(id(bad_grid[2]))

# Answer:
# Why did one assignment appear in all three rows?

In [ ]:
# EXERCISE 5 — BUILD INDEPENDENT ROWS
#
# Create a 3 x 4 grid of zeros in which every row is a different list.
# Then change grid[0][1] to 9 and verify that only one row changes.

# your code here

## Part 5 — Flat ↔ Grid Index Arithmetic
**Target time: 20 minutes**

A flat list and a grid can describe the same number of positions.

For a grid with `cols` columns:

- `(row, col) → flat index`: `idx = row * cols + col`
- `flat index → row`: `row = idx // cols`
- `flat index → col`: `col = idx % cols`

Think of `row * cols + col` as:

> skip whole rows, then move across the current row.

In [ ]:
# PREDICT BEFORE RUNNING
#
# This grid has 5 columns and 3 rows.
#
# 1. What flat index corresponds to row 2, column 3?
# 2. Where does flat index 13 land?

'''
   0    1    2    3    4
[
 ['a', 'b', 'c', 'd', 'e'],  0
 ['f', 'g', 'h', 'i', 'j'],  1
 ['k', 'l', 'm', 'n', 'o']   2
 ]

   0    1    2    3    4    5    6    7    8    9   10   11   12   13   14 
 ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o']
'''

cols = 5

row = 2
col = 3
idx = row * cols + col
print("flat index:", idx)

idx = 13
row = idx // cols
col = idx % cols
print("row:", row, "col:", col)

In [ ]:
# EXERCISE 6 — REBUILD A GRID
#
# Starting with the flat list below, rebuild a 3 x 4 grid.
#
# Requirement:
# For each row and column, compute:
#     idx = row * cols + col
#
# Do not slice the list.

flat = [11, 12, 13, 14, 21, 22, 23, 24, 31, 32, 33, 34]
rows = 3
cols = 4

rebuilt = []

# your code here

print(rebuilt)

### `%` as wrap-around

The remainder operator can also keep a growing value inside a fixed range.

If a list has length 4, `n % 4` can only be `0, 1, 2, 3`.

That makes `%` useful for repeating cycles.

In [ ]:
# EXERCISE 7 — ROTATING STORE ASSIGNMENTS
#
# There are 4 stores and 11 delivery days.
# Assign one store to each day in repeating order.
#
# Day 1 Boston
# Day 2 Cambridge
# Day 3 Wellesley
# Day 4 Newton
# Day 5 Boston
# ...

stores = ["Boston", "Cambridge", "Wellesley", "Newton"]

# your code here

## Part 6 — Asking AI: What Did You Decide?
**Target time: 35 minutes**

Now connect representation to AI.

When you ask AI to write a program:

> **Every important decision your request does not make is a decision you have delegated to the AI.**

There are layers of control:

| Layer | Who decides |
|---|---|
| **Problem** — what needs to be solved | You |
| **Specification** — important behavioral and design decisions | You |
| **Implementation** — mechanics left open by the specification | AI |
| **Execution** — what the code actually does | Python |

The goal is **not** to specify every line of code.

The goal is to make the decisions that matter.

Typical decisions include:

- **representation** — list, dictionary, grid, index?
- **input** — where does data come from?
- **stopping** — when is the task finished?
- **bad input** — what is rejected and how?
- **edge cases** — empty input, zero values, ties, missing keys
- **rules** — rounding, limits, definitions
- **output** — print, return, mutate, or create a new object?

### Three requests that sound similar

**Request A**

> Ask the user for prices until they enter 0, then print the total.

**Request B**

> Ask the user for prices until they're done, then print the total.

**Request C**

> Ask the user for prices one at a time. An empty entry means they are done. If an entry is not a number, print `Not a number` and ask again. If a price is negative, print `Price can't be negative` and ask again. Keep a running total instead of storing the prices. When they are done, print the total as dollars with two decimal places.

A few extra words can remove many hidden decisions.

In [ ]:
'''
DISCUSS BEFORE CODING

1. Which decision does Request A make that Request B leaves open?
2. List decisions that Request A still leaves open.
3. List decisions Request C makes that Request A does not.
4. Which Request C decisions affect BEHAVIOR?
5. Which implementation details can still safely be left to AI?
'''

# your answers:

### Same request, two reasonable programs

Both programs below satisfy **Request B**.

Run each with the same sequence:

`5`, `2.50`, `abc`, `-3`, `nan`, `infinity` then try to finish.

Do not focus first on syntax. Focus on **behavior**.

In [ ]:
prices = []

while True:
    entry = input("Enter a price (or done to finish): ")
    if entry == "done":
        break
    prices.append(float(entry))

print("Total:", sum(prices))

In [ ]:
total = 0

while True:
    entry = input("Price (press Enter when finished): ")
    if entry == "":
        break
    try:
        price = float(entry)
    except ValueError:
        print("Not a number")
        continue
    if price < 0:
        print("Price can't be negative")
        continue
    total += price

print(f"Total: ${total:.2f}")

In [ ]:
'''
COMPARE THE PROGRAMS

1. List every difference in how the programs BEHAVE.
2. Which differences are design decisions?
3. Which differences are only implementation/style?
4. Which decisions did Request B actually make?
5. Rewrite Request B so the important behavior is no longer ambiguous.
'''

# your answers:

### Two habits for working with AI

**Habit 1 — Treat a clarifying question as evidence of a missing decision.**

If AI asks:

> "What should happen if the user enters text?"

do not merely answer the chat. Add the answer to the **specification**.

**Habit 2 — Ask for a decision audit.**

After AI produces code, ask:

> **List every decision you made that my specification didn't state.**

Then classify each decision:

- **matters to me** → add it to the specification
- **does not matter to me** → knowingly delegate it

A specification is useful when you know what you decided and what you deliberately left open.

## Part 7 — Capstone: Choose Before You Ask
**Target time: 15 minutes**

Northstar wants to analyze:

- 8 stores
- each store has a `store_id`, name, manager, and region
- each store has sales for 7 days

Management needs to:

1. print a store directory with manager and region
2. find the weekly total for each store
3. compare Tuesday sales across every store

Do **not** ask AI to write code yet.

In [ ]:

raw_data = [
    [101, "Boston", "Ana", "East", 1250, 1425, 1380, 1510, 1600, 1740, 1685],
    [102, "Cambridge", "Ben", "East", 980, 1100, 1050, 1200, 1275, 1310, 1290],
    [103, "Wellesley", "Chen", "East", 1430, 1500, 1475, 1550, 1625, 1700, 1660],
    [104, "Newton", "Dana", "East", 1110, 1185, 1200, 1240, 1300, 1360, 1325],
    [105, "Providence", "Elena", "South", 1320, 1390, 1410, 1460, 1500, 1575, 1540],
    [106, "Hartford", "Farah", "South", 890, 940, 975, 1010, 1060, 1125, 1090],
    [107, "Albany", "Gabe", "North", 1040, 1095, 1130, 1175, 1210, 1260, 1235],
    [108, "Portland", "Hana", "North", 1175, 1240, 1210, 1290, 1340, 1400, 1375],
]




# CAPSTONE — DESIGN + SPECIFICATION
#
# A. Propose a representation.
# B. Name at least one reasonable alternative.
# C. Explain what your representation makes easy.
# D. State at least one invariant.

#
# Then write ONLY the representation/behavior portion of an AI specification
# for a function that returns Tuesday sales for every store.
#
# Your specification must state:
# - input representation
# - return representation
# - whether the input may be mutated
# - what happens with empty input
# - whether the function prints
#
# Do not prescribe Python mechanics unless they matter.

# design:
#
# sample data:
#
# AI specification:

## End-of-Class Reflection

Before leaving, be able to explain:

- why parallel lists are fragile
- why a list of dictionaries behaves like records
- why a grid is naturally a list of lists
- why repeated mutable rows can alias
- how a flat index maps to `(row, column)`
- why representation should follow the questions you need to answer
- what happens when an AI specification leaves an important decision unstated
- the difference between a **design decision** and an **implementation detail**

> **Choose the shape first. Then make the important choices explicit.**